# Session 11 — SMOTE vs scale_pos_weight

Per `docs/Credit_Risk_Pipeline_Plan_v3.md` (buổi 11): add SMOTE for the *at-application*
model — the one the Streamlit form will actually call — compare it against session 10's
`scale_pos_weight`, and pick the winner **by PR-AUC**.

Only `at_application` is tested here, deliberately. The portfolio model isn't making an
approval decision, so which imbalance strategy it uses doesn't change any product
behaviour; spending the comparison on the model that ships is the point.

**The two strategies differ in kind, not degree:**

- `scale_pos_weight` (session 10) reweights the *loss*. No rows are invented; the same
  5,686 real positives are simply worth more.
- **SMOTE** synthesises new minority rows by interpolating between real positives and
  their nearest neighbours. The model trains on data that partly did not happen.

That difference is the whole reason to measure rather than assume.

In [1]:
import sys

import pandas as pd
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import average_precision_score, roc_auc_score
from xgboost import XGBClassifier

sys.path.insert(0, '../etl')
sys.path.insert(0, '..')
from config import get_engine
from model.features import TARGET_COL, get_feature_columns, split_numeric_categorical
from model.preprocessing import build_pipeline

engine = get_engine()
df = pd.read_sql("SELECT * FROM ml_features", engine)

cols = get_feature_columns(df.columns, "at_application")
numeric_cols, categorical_cols = split_numeric_categorical(cols)

train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df[TARGET_COL]
)
neg, pos = train_df[TARGET_COL].value_counts().sort_index()
scale_pos_weight = neg / pos
print(f"{len(cols)} features | train {len(train_df)} | test {len(test_df)}")
print(f"scale_pos_weight = {scale_pos_weight:.4f}")

18 features | train 26064 | test 6517
scale_pos_weight = 3.5839


## Expected result — written before running

Session 10's lesson was that writing this down is worth it precisely *because* it can be
wrong (the predicted 0.82-0.84 came in at 0.8907, which forced a real investigation
instead of a shrug). Same discipline here.

**Expectation**: the two strategies land within ~0.01-0.02 PR-AUC of each other, with
`scale_pos_weight` slightly ahead or tied. Reasoning: session 8 found 9 of these 18
features are statistically noise, and SMOTE interpolates *between* points in that feature
space — synthesising along dimensions that carry no signal mostly manufactures plausible-
looking rows that teach the model nothing. SMOTE tends to help most when the minority
class is both small and cleanly separable; here it's neither (21.8% is imbalanced but not
extreme, and the classes overlap heavily).

**If SMOTE wins by a wide margin (>0.05 PR-AUC)**: be suspicious before celebrating —
check that the sampler really is inside the pipeline and therefore absent from the scored
test set. That exact mistake inflates SMOTE results in a lot of published notebooks.

**Decision rule, fixed now so the result can't move it**: PR-AUC on the held-out test set
decides, with 5-fold CV as the tiebreaker if the gap is under 0.01. PR-AUC rather than
ROC-AUC because it ignores true negatives, and 78.2% of this data is negative.

## Train both strategies

Both use the same estimator settings as session 10 (still untuned), the same split, and
the same `build_preprocessor` — so the *only* difference between the two rows below is how
class imbalance is handled.

Note `scale_pos_weight` is deliberately **not** set on the SMOTE variant: SMOTE already
balances the training set by adding rows, so reweighting the loss on top would correct for
the same imbalance twice and over-predict the positive class.

In [2]:
def xgb(**kwargs):
    return XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        eval_metric="logloss", random_state=42, **kwargs,
    )

strategies = {
    "scale_pos_weight": build_pipeline(
        numeric_cols, categorical_cols,
        estimator=xgb(scale_pos_weight=scale_pos_weight),
    ),
    "smote": build_pipeline(
        numeric_cols, categorical_cols,
        estimator=xgb(),
        sampler=SMOTE(random_state=42),
    ),
}

results = {}
for name, pipeline in strategies.items():
    pipeline.fit(train_df[cols], train_df[TARGET_COL])
    proba = pipeline.predict_proba(test_df[cols])[:, 1]
    results[name] = {
        "roc_auc": roc_auc_score(test_df[TARGET_COL], proba),
        "pr_auc": average_precision_score(test_df[TARGET_COL], proba),
    }

holdout = pd.DataFrame(results).T
holdout

,roc_auc,pr_auc
scale_pos_weight,0.890672,0.805474
smote,0.870686,0.779558


## Cross-validate before deciding

Session 10 found XGBoost's fold-to-fold spread on this data is wide (±0.0355 ROC-AUC), so
a holdout gap smaller than that spread isn't a real difference. Running 5-fold CV on both
strategies — with the sampler inside the pipeline, so SMOTE is re-fit per training fold
and never touches the fold being scored.

In [3]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_rows = {}
for name, pipeline in strategies.items():
    pr = cross_val_score(pipeline, df[cols], df[TARGET_COL], cv=cv, scoring="average_precision")
    roc = cross_val_score(pipeline, df[cols], df[TARGET_COL], cv=cv, scoring="roc_auc")
    cv_rows[name] = {
        "pr_auc_mean": pr.mean(), "pr_auc_std": pr.std(),
        "roc_auc_mean": roc.mean(), "roc_auc_std": roc.std(),
    }
    print(f"{name:18} PR folds {pr.round(4)}")

cv_results = pd.DataFrame(cv_rows).T
cv_results

scale_pos_weight   PR folds [0.7992 0.8168 0.8044 0.7879 0.8052]


smote              PR folds [0.7719 0.7932 0.7757 0.7677 0.7813]


,pr_auc_mean,pr_auc_std,roc_auc_mean,roc_auc_std
scale_pos_weight,0.802711,0.009389,0.892307,0.004840
smote,0.777965,0.008831,0.871462,0.005456


## Decision: `scale_pos_weight` wins — keep it, drop SMOTE

| Strategy | Holdout PR-AUC | 5-fold CV PR-AUC | Holdout ROC-AUC | CV ROC-AUC |
|---|---|---|---|---|
| **`scale_pos_weight`** | **0.8055** | **0.8027 ± 0.0094** | **0.8907** | **0.8923 ± 0.0048** |
| SMOTE | 0.7796 | 0.7780 ± 0.0088 | 0.8707 | 0.8715 ± 0.0055 |

Against the rule fixed before running:

- **Gap on held-out PR-AUC: 0.0259**, in `scale_pos_weight`'s favour.
- **Is it bigger than the fold spread?** Yes, comfortably — the gap is ~2.8× the CV
  standard deviation, and the two fold sets don't overlap at all
  (`scale_pos_weight` 0.7879–0.8168 vs SMOTE 0.7677–0.7932). Every single fold prefers
  `scale_pos_weight`. This is a real difference, not sampling noise.
- **Did the "SMOTE wins big" tripwire fire?** No — it lost, so there was no inflated
  result to go hunting for a leak behind.

The prediction written before running was "within ~0.01-0.02, `scale_pos_weight` slightly
ahead or tied." Direction correct, magnitude slightly underestimated (0.026 rather than
0.02). Close enough that the reasoning behind it — SMOTE interpolating across 9 noise
features manufactures rows that teach nothing — is probably the right explanation, though
this comparison alone doesn't isolate that as the cause.

**Decision**: the at-application model keeps `scale_pos_weight`. It wins on the metric
fixed in advance, and it's also the easier thing to defend — it trains on real rows only,
where SMOTE ships a model partly fitted to interpolated data that never happened.

---

## Correction to session 10's cross-validation claim

Running CV here surfaced an error in session 10's own analysis, worth recording rather
than quietly fixing.

Session 10 reported `at_application` XGBoost at **0.8606 ± 0.0355** ROC-AUC and concluded
the single holdout number (0.8907) "sits on the optimistic side of a wide distribution",
with fold-to-fold variance "~5× the Logistic Regression baseline". That used
`cross_val_score(..., cv=5)`, which splits **without shuffling**.

The rows of `ml_features` are not randomly ordered with respect to the target. Default
rate across five contiguous blocks of the table:

```
block 0: 27.8%   block 1: 19.1%   block 2: 24.3%   block 3: 18.0%   block 4: 20.0%
```

`ml_features` has no `ORDER BY`, so rows come back in heap order ≈ insertion order ≈ the
original spreadsheet's order, and that order carries structure. Unshuffled folds therefore
train and test on systematically different populations, and the resulting spread measures
**the table's row ordering, not the model's stability**.

With `StratifiedKFold(shuffle=True)`:

| | unshuffled `cv=5` (session 10) | shuffled (correct) |
|---|---|---|
| XGBoost `at_application` ROC-AUC | 0.8606 ± 0.0355 | **0.8923 ± 0.0048** |

Two conclusions flip:

1. **The holdout number was fine all along.** 0.8907 holdout vs 0.8923 shuffled CV mean —
   they agree to within 0.002. It was never "on the optimistic side" of anything.
2. **XGBoost is not unstable here.** Its shuffled spread (±0.0048) is actually *tighter*
   than the Logistic Regression baseline's (±0.0083 on the same feature set). The "5×
   more variance" claim was an artifact of the measurement, not a property of the model.

The original instinct — don't quote one holdout number without checking it — was right,
and checking is exactly what caught this. But the check itself was wrong the first time,
which is its own lesson: `cross_val_score(cv=5)` silently means unshuffled, and on any
dataset whose row order isn't random that measures the wrong thing. Pass an explicit
`StratifiedKFold(shuffle=True, random_state=...)`.